# Домашка
1. Дособрать TabNet (https://arxiv.org/pdf/1908.07442.pdf)
2. Запустить на датасете income
3. Залогировать метрики с помощью tensorboard/mlflow

In [1]:
!pip install torchmetrics tensorboard --quiet
%load_ext tensorboard
!rm -rf runs

In [2]:
import math

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

from torchmetrics import Accuracy


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## CustomDataset

In [3]:
class CustomDataset(Dataset):
    def __init__(
        self,
        path: str,
        num_cols: list[str],
        log1p_cols: list[str],
        n_categories_of: dict[str, int],
    ) -> None:

        data = pd.read_parquet(path)

        cat_cols = list(n_categories_of.keys())

        X = data[num_cols + cat_cols].reset_index(drop=True)

        for col in num_cols:
            if X[col].isna().any():
                X[col] = X[col].fillna(X[col].median())

        for col in log1p_cols:
            X[col] = np.log1p(X[col])

        self.X = X.to_numpy(dtype=np.float32)
        self.y = np.asarray(
            [0 if el == "<50k" else 1 for el in data["salary"].values],
            dtype=np.float32,
        )

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        features = torch.from_numpy(self.X[idx])
        label = torch.from_numpy(self.y[idx:idx+1])
        return features, label

## Архитектура

In [4]:
class DenseFeatureLayer(nn.Module):
    def __init__(self, num_cols: list[str], n_categories_of: dict[str, int]) -> None:
        super().__init__()

        self.num_cols = num_cols
        self.cat_cols = list(n_categories_of.keys())

        self.embeddings = nn.ModuleDict()
        self.output_size = len(self.num_cols)
        for col in self.cat_cols:
            n_categories = n_categories_of[col]
            embedding_size = min(4, round(1.6 * n_categories ** 0.56))
            self.embeddings[col] = nn.Embedding(n_categories, embedding_size)
            self.output_size += embedding_size

        self.feature_bn = nn.BatchNorm1d(self.output_size)

    def init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            # module.bias.data.fill_(0.001)

    def forward(self, x: Tensor) -> Tensor:

        num_feats, cat_feats = torch.split(x, [len(self.num_cols), len(self.cat_cols)], dim=1)
        cat_feats = cat_feats.long()

        embs = [self.embeddings[col](cat_feats[:, i]) for i, col in enumerate(self.cat_cols)]
        emb_output = torch.cat(embs, dim=1)

        concat_input = torch.cat([num_feats, emb_output], dim=1)
        output = self.feature_bn(concat_input)

        return output

In [5]:
class GluBlock(nn.Module):
    def __init__(
        self,
        in_size: int,
        out_size: int | None = None,
        fc: nn.Linear | None = None,
    ) -> None:
        super().__init__()
        out_size = out_size if out_size is not None else in_size
        out_size *= 2
        self.fc = fc or nn.Linear(in_size, out_size)
        self.bn = nn.BatchNorm1d(out_size)

    def init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            # module.bias.data.fill_(0.001)

    def forward(self, x: Tensor) -> Tensor:
        h = self.fc(x)
        h = self.bn(h)
        h = F.glu(h, dim=-1)
        return h

In [6]:
class FeatureTransformer(nn.Module):
    def __init__(self, n_glu_blocks: int, size: int, scale: float, shared_fc: nn.Linear) -> None:
        super().__init__()

        glu_blocks = [GluBlock(size, fc=shared_fc)] + [GluBlock(size) for _ in range(n_glu_blocks)]
        self.glu_blocks = nn.ModuleList(glu_blocks)
        self.scale = scale

    def forward(self, x: Tensor) -> Tensor:
        h = x
        for i, block in enumerate(self.glu_blocks):
            if i == 0:
                h = block(h)
            else:
                h = (h + block(h)) * self.scale
        return h

In [7]:
class Split(nn.Module):
    def __init__(self, n_d: int, n_a: int) -> None:
        super().__init__()
        self.n_d = n_d
        self.n_a = n_a

    def forward(self, x: Tensor) -> tuple[Tensor, Tensor]:
        d, a = torch.split(x, [self.n_d, self.n_a], dim=-1)
        return d, a

In [8]:
class Sparsemax(nn.Module):
    def __init__(self, dim: int | None = None) -> None:
        """
        Args:
            dim: The dimension over which to apply the sparsemax function.
        """
        super().__init__()

        self.dim = dim if dim is not None else -1

    def forward(self, x: Tensor) -> Tensor:
        """
        Args:
            x: Input tensor. First dimension should be the batch size.
        Returns:
            Output tensor of shape [batch_size x n_logits].
        """
        # Sparsemax currently only handles 2-dim tensors,
        # so we reshape to a convenient shape and reshape back after sparsemax
        h = x.transpose(0, self.dim)
        original_size = h.size()
        h = h.reshape(h.size(0), -1)
        h = h.transpose(0, 1)
        dim = 1

        n_logits = h.size(dim)

        # Translate h by max for numerical stability
        h = h - torch.max(h, dim=dim, keepdim=True).values.expand_as(h)

        # Sort h in descending order.
        zs = torch.sort(h, dim=dim, descending=True).values
        range = torch.arange(start=1, end=n_logits + 1, step=1, device=DEVICE, dtype=h.dtype).view(1, -1)
        range = range.expand_as(zs)

        # Determine sparsity of projection
        bound = 1 + range * zs
        cumulative_sum_zs = torch.cumsum(zs, dim)
        is_gt = torch.gt(bound, cumulative_sum_zs).type(h.dtype)
        k = torch.max(is_gt * range, dim=dim, keepdim=True).values

        # Compute threshold function
        zs_sparse = is_gt * zs

        # Compute taus
        taus = (torch.sum(zs_sparse, dim=dim, keepdim=True) - 1) / k
        taus = taus.expand_as(h)

        # Sparsemax
        self.output = torch.max(torch.zeros_like(h), h - taus)

        # Reshape back to original shape
        output = self.output
        output = output.transpose(0, 1)
        output = output.reshape(original_size)
        output = output.transpose(0, self.dim)
        return output

    def backward(self, grad_output):
        dim = 1
        nonzeros = torch.ne(self.output, 0)
        amount = torch.sum(grad_output * nonzeros, dim=dim) / torch.sum(nonzeros, dim=dim)
        self.grad_input = nonzeros * (grad_output - amount.expand_as(grad_output))
        return self.grad_input

In [9]:
class AttentiveTransformer(nn.Module):
    def __init__(self, n_a: int, n_features: int) -> None:
        super().__init__()
        self.fc = nn.Linear(n_a, n_features)
        self.bn = nn.BatchNorm1d(n_features)
        self.sparsemax = Sparsemax()

    def init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            # module.bias.data.fill_(0.001)

    def forward(self, x: Tensor, prior: Tensor) -> Tensor:
        h = self.fc(x)
        h = self.bn(h)
        h = h * prior
        h = self.sparsemax(h)
        return h

In [10]:
class TabNet(nn.Module):
    def __init__(
        self,
        num_cols: list[str],
        n_categories_of: dict[str, int],
        n_classes: int,
        n_d: int = 16,
        n_a: int = 16,
        gamma: float = 1.5,
        glu_scale: float = 0.5,
        n_specific_glu_blocks: int = 2,
        n_steps: int = 6,
    ) -> None:
        super().__init__()

        self.dense_feature_layer = DenseFeatureLayer(num_cols, n_categories_of)

        glu_size = n_d + n_a
        self.n_d = n_d
        self.gamma = gamma
        self.glu_scale = math.sqrt(glu_scale)
        self.n_steps = n_steps

        self.shared_fc = nn.Linear(self.dense_feature_layer.output_size, 2 * glu_size)

        self.feature_transformers = nn.ModuleList([
            FeatureTransformer(n_specific_glu_blocks, glu_size, glu_scale, self.shared_fc)
            for _ in range(n_steps + 1)
        ])

        self.attentive_transformers = nn.ModuleList([
            AttentiveTransformer(n_a, self.dense_feature_layer.output_size)
            for _ in range(n_steps)
        ])

        self.split = Split(n_d, n_a)

        self.final_layer = nn.Linear(n_d, n_classes)

    def init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            # module.bias.data.fill_(0.001)

    def forward(self, x: Tensor) -> Tensor:

        x = self.dense_feature_layer(x)

        h = self.feature_transformers[0](x)
        _, a = self.split(h)

        prior = torch.ones_like(x)
        d_out = torch.zeros(x.size(0), self.n_d, device=DEVICE)

        for i in range(self.n_steps):
            mask = self.attentive_transformers[i](a, prior)
            prior = prior * (self.gamma - mask)
            x_masked = mask * x
            h = self.feature_transformers[i+1](x_masked)
            d, a = self.split(h)
            d = F.relu(d)
            d_out += d

        out = self.final_layer(d_out)

        return out

## Train Loop

In [11]:
TRAIN_DATA_PATH = "train_adult.parquet"
VALID_DATA_PATH = "valid_adult.parquet"
N_CATEGORIES_OF = {
    "workclass_cat": 9,
    "education_cat": 16,
    "marital-status_cat": 7,
    "occupation_cat": 16,
    "relationship_cat": 6,
    "race_cat": 5,
    "sex_cat": 2,
    "native-country_cat": 42,
}
NUM_COLS = [
    "age",
    "fnlwgt",
    "education-num",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
]
LOG1P_COLS = ["fnlwgt", "capital-gain", "capital-loss"]

N_CLASSES = 1

EPOCHS = 50
LEARNING_RATE = 3e-4
BATCH_SIZE = 512

In [12]:
train_dataset = CustomDataset(TRAIN_DATA_PATH, NUM_COLS, LOG1P_COLS, N_CATEGORIES_OF)
train_dataloader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True)

valid_dataset = CustomDataset(VALID_DATA_PATH, NUM_COLS, LOG1P_COLS, N_CATEGORIES_OF)
valid_dataloader = DataLoader(valid_dataset, BATCH_SIZE)

In [13]:
class Tab:
    def __init__(
        self,
        num_cols: list[str],
        n_categories_of: dict[str, int],
        n_classes: int,
        learning_rate: float,
    ) -> None:
        self.net = TabNet(num_cols, n_categories_of, n_classes)
        self.loss = torch.nn.BCEWithLogitsLoss()
        self.train_accuracy = Accuracy(task="binary")
        self.valid_accuracy = Accuracy(task="binary")
        self.optimizer = optim.Adam(self.net.parameters(), lr=learning_rate, weight_decay=1e-6)
        self.train_writer = SummaryWriter("runs/train")
        self.valid_writer = SummaryWriter("runs/valid")

    def fit(self, train_loader: DataLoader, valid_loader: DataLoader, epochs: int) -> None:
        step = 1
        for epoch in range(1, epochs + 1):
            self.net.train()
            self.train_accuracy.reset()
            total_loss = 0.0
            n = 0
            for features, labels in train_loader:
                self.optimizer.zero_grad()
                out = self.net(features)
                loss = self.loss(out, labels)
                out = torch.sigmoid(out)
                loss.backward()
                self.optimizer.step()

                self.train_accuracy(out, labels)
                total_loss += loss.item()
                n += 1

                self.train_writer.add_scalar("CrossEntropyLoss", loss, step)

                step += 1

            train_loss = total_loss / n
            train_acc = self.train_accuracy.compute().item()

            self.train_writer.add_scalar("Accuracy", train_acc, step)


            self.net.eval()
            self.valid_accuracy.reset()
            total_loss = 0.0
            n = 0
            with torch.no_grad():
                for features, labels in valid_loader:
                    out = self.net(features)
                    loss = self.loss(out, labels)
                    out = torch.sigmoid(out)
                    self.valid_accuracy(out, labels)
                    total_loss += loss.item()
                    n += 1

            valid_loss = total_loss / n
            valid_acc = self.valid_accuracy.compute().item()

            self.valid_writer.add_scalar("CrossEntropyLoss", valid_loss, step)
            self.valid_writer.add_scalar("Accuracy", valid_acc, step)

            print(f"{epoch=}, {train_loss=:.3f}, {valid_loss=:.3f}, {train_acc=:.3f}, {valid_acc=:.3f}")

In [14]:
tab = Tab(NUM_COLS, N_CATEGORIES_OF, N_CLASSES, LEARNING_RATE)
tab.fit(train_dataloader, valid_dataloader, EPOCHS)

epoch=1, train_loss=0.634, valid_loss=0.614, train_acc=0.721, valid_acc=0.764
epoch=2, train_loss=0.570, valid_loss=0.551, train_acc=0.767, valid_acc=0.766
epoch=3, train_loss=0.530, valid_loss=0.517, train_acc=0.769, valid_acc=0.767
epoch=4, train_loss=0.503, valid_loss=0.490, train_acc=0.771, valid_acc=0.769
epoch=5, train_loss=0.480, valid_loss=0.469, train_acc=0.773, valid_acc=0.776
epoch=6, train_loss=0.466, valid_loss=0.460, train_acc=0.775, valid_acc=0.778
epoch=7, train_loss=0.454, valid_loss=0.447, train_acc=0.779, valid_acc=0.779
epoch=8, train_loss=0.448, valid_loss=0.438, train_acc=0.782, valid_acc=0.783
epoch=9, train_loss=0.440, valid_loss=0.434, train_acc=0.785, valid_acc=0.789
epoch=10, train_loss=0.436, valid_loss=0.431, train_acc=0.787, valid_acc=0.789
epoch=11, train_loss=0.432, valid_loss=0.427, train_acc=0.790, valid_acc=0.790
epoch=12, train_loss=0.426, valid_loss=0.420, train_acc=0.793, valid_acc=0.790
epoch=13, train_loss=0.421, valid_loss=0.418, train_acc=0.798

In [15]:
# %tensorboard --logdir runs